# EVE 310 - Lab 11: Comparing regression models — activity

**Module 2 | 11/12/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab11-dairy-farm-challenge/notebooks/lab11-activity.ipynb)

## Learning objectives

Fit and score a linear model and a random forest. Then add a column built from units, and compare again.

<details><summary>More: the three objectives</summary>

By the end of this lab you will be able to:

1. Fit a linear model and a random forest on the same training farms. Say what the linear model's betas mean, with their units.
2. Score both models on the testing farms with the MAE. Compare them with the know-nothing model. Check the forest for overfitting.
3. Build a new column from the units of two other columns. Decide if it helps the linear model predict better than the forest.

</details>

## Before you start

1. Click **Copy to Drive** at the top. Work in the copy, or you lose your work.
2. Run the setup cell first.
3. Work in order and keep the given variable names. In Questions 1 and 2, replace each `--` with one line of code. In Question 3, replace each `--` with a whole step. Try first. If you are stuck, open **Show the finished cell**.
4. Keep `random_state=25` in the split and `random_state=0` in the forest, so your numbers match the autograder and quiz.
5. Upload this notebook to Gradescope. The upload gives feedback only and is **not graded**. You can upload many times. The Lab 11 Canvas quiz is graded and asks about your results. Finish the activity before the quiz, at home if needed.

<details><summary>More: setup and grading</summary>

Colab deletes work that is not in your Drive copy when the session ends. The setup cell downloads the two data files into `DATA_DIR`. It also creates `FIGURES_DIR`. The three questions follow the steps of the tutorial's plan. Question 1 fits two models. Question 2 checks them. Question 3 prepares a new column, then fits and checks a model with it. In Question 3, one `--` can stand for many lines. **Show the finished cell** under each question has the answer. The autograder gives feedback on each question.

</details>

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab11-dairy-farm-challenge"
DATA_FILES = ["power_demand.csv", "Dairy-Farm-Dataset.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**The data (run as is)**

`Dairy-Farm-Dataset.csv` has 10,000 dairy farms, made up for this lab. A dairy cooperative is a group of farms that work together. It asks which practices lower `Carbon_footprint`, and by how much.

<details><summary>More: emissions and units</summary>

Each row is one farm. Making milk releases greenhouse gases. The cows and their manure release methane. Fields with fertilizer release nitrous oxide. Growing feed and using energy release carbon dioxide. `Carbon_footprint` is the greenhouse gas a farm releases for each kg of milk. Milk here means fat-and-protein-corrected milk, FPCM. CO₂e means CO₂-equivalent. It adds up all the greenhouse gases. A gas that warms the climate more counts more.

</details>

| Column | Meaning | Units |
| --- | --- | --- |
| `Feed_inefficiency` | feed eaten | kg feed per kg milk |
| `Protein_efficiency` | part of the eaten nitrogen that goes into milk, growth and beef | % |
| `Animal_mortality` | deaths that year | % of the herd |
| `Fertilizer_use` | nitrogen put on the land as manure and fertilizer | kg N per hectare |
| `Land_use` | land to grow all feed | m² per kg milk |
| `Soy_use` | soy in feed | g per kg milk |
| `Permanent_grassland` | land kept as grass for years | hectares per cow |
| `Grazing` | grazing ≥ 6 h/day, 120 days/year | % of cows |
| `Manure_biogas` | 1 if 75%+ of manure goes to biogas or acid treatment | 0 or 1 |
| `Renewable_electricity_use` | 1 if all electricity is wind, solar, biogas or hydro | 0 or 1 |
| `Carbon_footprint` | to predict | kg CO₂e per kg milk |

70% of the farms are **training farms**, for fitting. 30% are **testing farms**, for scoring. `random_state=25` makes the split the same every time.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

farm_df = pd.read_csv(DATA_DIR / 'Dairy-Farm-Dataset.csv')
print(farm_df.shape)
# The mean, smallest and largest value of every column, one row per column
print(farm_df.describe().loc[['mean', 'min', 'max']].T.round(3))

y = farm_df['Carbon_footprint']
x = farm_df.drop(columns=['Carbon_footprint'])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=25)
print(len(x_train), 'training farms,', len(x_test), 'testing farms')

## Your turn

**Question 1 — Fit: a linear model and a forest on all ten columns**

Create each model and add `.fit(...)` on the same line, as in Lab 10's `KMeans(...).fit(X)`.

- `lm_all`: a `LinearRegression` fitted on `x_train` and `y_train`
- `betas`: its betas as a pandas Series, labeled by column name (`x_train.columns`), as in Lab 6
- `cut_both`: the predicted footprint change when a farm switches both `Manure_biogas` and `Renewable_electricity_use` from 0 to 1. Read both betas from `betas` by name.
- `rf`: a `RandomForestRegressor(random_state=0)` fitted on the same training farms. It takes 10 to 20 seconds.

The first check compares `lm_all` and `betas` with the normal equation on the 7,000 training farms. The second compares `cut_both` with one farm's predictions.

<details><summary>More: betas and checks</summary>

A beta is the change in the predicted footprint when its column goes up by 1 and the other columns stay the same. `cut_both` is in kg CO₂e per kg of milk, like the footprint. Its check takes one farm with both practices and the same farm with neither. The two predictions of `lm_all` must differ by `cut_both`. The normal equation uses the design matrix `X1`. It has 11 columns: the column of ones for the intercept, then the ten farm columns. A random forest averages the predictions of many decision trees.

</details>

In [ ]:
# A linear model on all ten columns, created and fitted in one line
lm_all = --
# Its betas, one per column, labeled by the column names
betas = --
# The predicted change for a farm that switches to biogas and to renewable electricity
cut_both = --
# A random forest on the same rows (keep random_state=0)
rf = --

print(betas.round(4))
print('intercept:', round(lm_all.intercept_, 4))
print('both practices:', round(cut_both, 4), 'kg CO2e per kg of milk')

# The normal equation for the 7,000 training farms: a column of ones, then the ten columns
X1 = np.c_[np.ones(len(x_train)), x_train]
beta_check = np.dot(np.linalg.inv(np.dot(X1.T, X1)), np.dot(X1.T, y_train))
print('The normal equation gives the intercept and the betas, labeled by column?',
      np.isclose(beta_check[0], lm_all.intercept_) and np.allclose(beta_check[1:], betas)
      and list(getattr(betas, 'index', [])) == list(x_train.columns))
# One farm with both practices, and the same farm with neither
farm_both = x_train.iloc[[0]].assign(Manure_biogas=1, Renewable_electricity_use=1)
farm_neither = x_train.iloc[[0]].assign(Manure_biogas=0, Renewable_electricity_use=0)
print("cut_both is the change in that farm's prediction, and the forest saw the ten columns with random_state=0?",
      np.ndim(cut_both) == 0 and np.isclose(cut_both, lm_all.predict(farm_both)[0] - lm_all.predict(farm_neither)[0])
      and getattr(rf, 'n_features_in_', 0) == 10 and rf.random_state == 0)

<details><summary>Show the finished cell</summary>

```python
# A linear model on all ten columns, created and fitted in one line
lm_all = LinearRegression().fit(x_train, y_train)
# Its betas, one per column, labeled by the column names
betas = pd.Series(lm_all.coef_, index=x_train.columns)
# The predicted change for a farm that switches to biogas and to renewable electricity
cut_both = betas['Manure_biogas'] + betas['Renewable_electricity_use']
# A random forest on the same rows (keep random_state=0)
rf = RandomForestRegressor(random_state=0).fit(x_train, y_train)

print(betas.round(4))
print('intercept:', round(lm_all.intercept_, 4))
print('both practices:', round(cut_both, 4), 'kg CO2e per kg of milk')

# The normal equation for the 7,000 training farms: a column of ones, then the ten columns
X1 = np.c_[np.ones(len(x_train)), x_train]
beta_check = np.dot(np.linalg.inv(np.dot(X1.T, X1)), np.dot(X1.T, y_train))
print('The normal equation gives the intercept and the betas, labeled by column?',
      np.isclose(beta_check[0], lm_all.intercept_) and np.allclose(beta_check[1:], betas)
      and list(getattr(betas, 'index', [])) == list(x_train.columns))
# One farm with both practices, and the same farm with neither
farm_both = x_train.iloc[[0]].assign(Manure_biogas=1, Renewable_electricity_use=1)
farm_neither = x_train.iloc[[0]].assign(Manure_biogas=0, Renewable_electricity_use=0)
print("cut_both is the change in that farm's prediction, and the forest saw the ten columns with random_state=0?",
      np.ndim(cut_both) == 0 and np.isclose(cut_both, lm_all.predict(farm_both)[0] - lm_all.predict(farm_neither)[0])
      and getattr(rf, 'n_features_in_', 0) == 10 and rf.random_state == 0)
```

</details>

Which of the two practices cuts the footprint more, according to `betas`? Why can you add their betas for a farm that does both?

*Your answer:*

**Question 2 — Check: score both models on the testing farms**

Score the given predictions with the MAE, the average size of the errors. Put the truth first: `mean_absolute_error(y_test, ...)` or `np.mean(np.abs(...))`.

- `mae_lm`, `mae_rf`: the testing MAE of each model
- `mae_train_rf`: the forest's training MAE
- `mae_flat`: the testing MAE of the **know-nothing model**. It predicts the training farms' mean footprint for every testing farm.

The check computes each MAE by hand.

<details><summary>More: predictions and overfitting</summary>

`y_pred_...` are the predictions on the testing farms. `y_fit_...` are the predictions on the training farms. The testing MAE is the MAE on the testing farms. The training MAE is the MAE on the training farms. Every MAE is in kg CO₂e per kg of milk. The cell also prints the linear model's training MAE, to compare. A model **overfits** when its training MAE is much smaller than its testing MAE. It has partly memorized its training farms, so it does worse on farms it has not seen.

</details>

In [ ]:
# Predictions on the testing farms and on the training farms, made for you
y_pred_lm = lm_all.predict(x_test)
y_pred_rf = rf.predict(x_test)
y_fit_lm = lm_all.predict(x_train)
y_fit_rf = rf.predict(x_train)

# The testing MAE of each model, in kg CO2e per kg of milk
mae_lm = --
mae_rf = --
# The forest's MAE on the farms it was fitted on
mae_train_rf = --
# A model that knows nothing: every testing farm predicted with the mean of the training farms
mae_flat = --

print('knows nothing: MAE testing', round(mae_flat, 4))
print('linear:        MAE testing', round(mae_lm, 4), '  training', round(mean_absolute_error(y_train, y_fit_lm), 4))
print('forest:        MAE testing', round(mae_rf, 4), '  training', round(mae_train_rf, 4))
print('Each MAE is one number, the mean absolute error on the right farms?',
      all(np.ndim(v) == 0 for v in (mae_lm, mae_rf, mae_train_rf, mae_flat))
      and np.isclose(mae_lm, np.mean(np.abs(y_test - y_pred_lm))) and np.isclose(mae_rf, np.mean(np.abs(y_test - y_pred_rf)))
      and np.isclose(mae_train_rf, np.mean(np.abs(y_train - y_fit_rf)))
      and np.isclose(mae_flat, np.mean(np.abs(y_test - np.mean(y_train)))))

# Predicted against true footprint for the testing farms; on the dashed line, prediction = truth
fig_pred, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(y_test, y_pred_lm, s=4)
ax1.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax1.set_title('Linear model, ten columns')
ax2.scatter(y_test, y_pred_rf, s=4, color='tab:orange')
ax2.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax2.set_title('Random forest, ten columns')
for ax in (ax1, ax2):
    ax.set_xlabel('true footprint (kg CO2e per kg milk)')
    ax.set_ylabel('predicted footprint')
fig_pred.savefig(FIGURES_DIR / 'lab11_predicted_vs_true.png', dpi=150)

<details><summary>Show the finished cell</summary>

```python
# Predictions on the testing farms and on the training farms, made for you
y_pred_lm = lm_all.predict(x_test)
y_pred_rf = rf.predict(x_test)
y_fit_lm = lm_all.predict(x_train)
y_fit_rf = rf.predict(x_train)

# The testing MAE of each model, in kg CO2e per kg of milk
mae_lm = mean_absolute_error(y_test, y_pred_lm)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
# The forest's MAE on the farms it was fitted on
mae_train_rf = mean_absolute_error(y_train, y_fit_rf)
# A model that knows nothing: every testing farm predicted with the mean of the training farms
mae_flat = np.mean(np.abs(y_test - np.mean(y_train)))

print('knows nothing: MAE testing', round(mae_flat, 4))
print('linear:        MAE testing', round(mae_lm, 4), '  training', round(mean_absolute_error(y_train, y_fit_lm), 4))
print('forest:        MAE testing', round(mae_rf, 4), '  training', round(mae_train_rf, 4))
print('Each MAE is one number, the mean absolute error on the right farms?',
      all(np.ndim(v) == 0 for v in (mae_lm, mae_rf, mae_train_rf, mae_flat))
      and np.isclose(mae_lm, np.mean(np.abs(y_test - y_pred_lm))) and np.isclose(mae_rf, np.mean(np.abs(y_test - y_pred_rf)))
      and np.isclose(mae_train_rf, np.mean(np.abs(y_train - y_fit_rf)))
      and np.isclose(mae_flat, np.mean(np.abs(y_test - np.mean(y_train)))))

# Predicted against true footprint for the testing farms; on the dashed line, prediction = truth
fig_pred, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(y_test, y_pred_lm, s=4)
ax1.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax1.set_title('Linear model, ten columns')
ax2.scatter(y_test, y_pred_rf, s=4, color='tab:orange')
ax2.plot([0.7, 2.1], [0.7, 2.1], color='k', ls='dashed')
ax2.set_title('Random forest, ten columns')
for ax in (ax1, ax2):
    ax.set_xlabel('true footprint (kg CO2e per kg milk)')
    ax.set_ylabel('predicted footprint')
fig_pred.savefig(FIGURES_DIR / 'lab11_predicted_vs_true.png', dpi=150)
```

</details>

What percent of the know-nothing model's error does each model remove? Which model overfits, and how do its two MAEs show it? Does the overfitting change which model predicts the testing farms better?

*Your answer:*

**Question 3 — Prepare, fit, check: one new column from the units**

Nitrous oxide from fertilized fields depends on the nitrogen put on the land **per kg of milk**. `Fertilizer_use` is in kg N per hectare. `Land_use` is in m² per kg milk. Multiply them and convert m² to hectares: 1 hectare = 10,000 m². You get kg N per kg milk.

- `x_train_n`, `x_test_n`: copies of `x_train` and `x_test`, made for you. Add `N_per_kg_milk` to both, so each has eleven columns. `x_train` and `x_test` keep ten.
- `lm_n`: a `LinearRegression` fitted on `x_train_n`
- `beta_n`: its beta for `N_per_kg_milk`
- `mae_n`: its testing MAE

Write each step under its comment. The units check prints False if your column is 10 or more times too big or too small. If the first check prints False, you may have added the column to `x_train` or `x_test`. Run the data cell again to rebuild the split. Then run this cell.

In [ ]:
# Copies of the split, made for you: x_train and x_test keep their ten columns
x_train_n = x_train.copy()
x_test_n = x_test.copy()

# Each placeholder line below is a whole step: replace it with as many lines as the step needs

# Step 1, prepare: add N_per_kg_milk, kg of nitrogen per kg of milk, to x_train_n and to x_test_n
--

# Step 2, fit: lm_n, a linear model fitted on x_train_n, and beta_n, its beta for N_per_kg_milk
--

# Step 3, check: mae_n, the testing MAE of lm_n
--

# Step 4: the checks and the three models side by side, written for you
has_n = 'N_per_kg_milk' in x_train_n.columns and 'N_per_kg_milk' in x_test_n.columns
print('N_per_kg_milk in both copies, eleven columns each, ten still in x_train and x_test?',
      has_n and x_train_n.shape[1] == 11 and x_test_n.shape[1] == 11 and x_train.shape[1] == 10 and x_test.shape[1] == 10)
print('N_per_kg_milk in kg of nitrogen per kg of milk (every value between 0.001 and 0.5)?',
      has_n and bool(x_train_n['N_per_kg_milk'].between(0.001, 0.5).all() and x_test_n['N_per_kg_milk'].between(0.001, 0.5).all()))
# A least-squares fit leaves X-transpose times its residuals at zero (tutorial section 2), on the rows it was fitted on
print('lm_n fitted on the training farms, beta_n its N_per_kg_milk beta, and mae_n its testing MAE?',
      has_n and np.ndim(beta_n) == 0 and np.ndim(mae_n) == 0
      and np.allclose(np.dot(x_train_n.T, y_train - lm_n.predict(x_train_n)), 0, atol=1e-3)
      and np.isclose(beta_n, lm_n.coef_[list(x_train_n.columns).index('N_per_kg_milk')])
      and np.isclose(mae_n, np.mean(np.abs(y_test - lm_n.predict(x_test_n)))))
scores_df = pd.DataFrame({'model': ['linear, ten columns', 'random forest, ten columns', 'linear, eleven columns'],
                          'MAE testing': [mae_lm, mae_rf, mae_n],
                          'R2 testing': [r2_score(y_test, y_pred_lm), r2_score(y_test, y_pred_rf),
                                         r2_score(y_test, lm_n.predict(x_test_n))]})
print(scores_df.round(4))
print('beta for N_per_kg_milk:', round(beta_n, 3))

<details><summary>Show the finished cell</summary>

```python
# Copies of the split, made for you: x_train and x_test keep their ten columns
x_train_n = x_train.copy()
x_test_n = x_test.copy()

# Each placeholder line below is a whole step: replace it with as many lines as the step needs

# Step 1, prepare: add N_per_kg_milk, kg of nitrogen per kg of milk, to x_train_n and to x_test_n
x_train_n['N_per_kg_milk'] = x_train_n['Fertilizer_use'] * x_train_n['Land_use'] / 10000
x_test_n['N_per_kg_milk'] = x_test_n['Fertilizer_use'] * x_test_n['Land_use'] / 10000

# Step 2, fit: lm_n, a linear model fitted on x_train_n, and beta_n, its beta for N_per_kg_milk
lm_n = LinearRegression().fit(x_train_n, y_train)
beta_n = pd.Series(lm_n.coef_, index=x_train_n.columns)['N_per_kg_milk']

# Step 3, check: mae_n, the testing MAE of lm_n
mae_n = mean_absolute_error(y_test, lm_n.predict(x_test_n))

# Step 4: the checks and the three models side by side, written for you
has_n = 'N_per_kg_milk' in x_train_n.columns and 'N_per_kg_milk' in x_test_n.columns
print('N_per_kg_milk in both copies, eleven columns each, ten still in x_train and x_test?',
      has_n and x_train_n.shape[1] == 11 and x_test_n.shape[1] == 11 and x_train.shape[1] == 10 and x_test.shape[1] == 10)
print('N_per_kg_milk in kg of nitrogen per kg of milk (every value between 0.001 and 0.5)?',
      has_n and bool(x_train_n['N_per_kg_milk'].between(0.001, 0.5).all() and x_test_n['N_per_kg_milk'].between(0.001, 0.5).all()))
# A least-squares fit leaves X-transpose times its residuals at zero (tutorial section 2), on the rows it was fitted on
print('lm_n fitted on the training farms, beta_n its N_per_kg_milk beta, and mae_n its testing MAE?',
      has_n and np.ndim(beta_n) == 0 and np.ndim(mae_n) == 0
      and np.allclose(np.dot(x_train_n.T, y_train - lm_n.predict(x_train_n)), 0, atol=1e-3)
      and np.isclose(beta_n, lm_n.coef_[list(x_train_n.columns).index('N_per_kg_milk')])
      and np.isclose(mae_n, np.mean(np.abs(y_test - lm_n.predict(x_test_n)))))
scores_df = pd.DataFrame({'model': ['linear, ten columns', 'random forest, ten columns', 'linear, eleven columns'],
                          'MAE testing': [mae_lm, mae_rf, mae_n],
                          'R2 testing': [r2_score(y_test, y_pred_lm), r2_score(y_test, y_pred_rf),
                                         r2_score(y_test, lm_n.predict(x_test_n))]})
print(scores_df.round(4))
print('beta for N_per_kg_milk:', round(beta_n, 3))
```

</details>

Say what `beta_n` means in words, with units. Suppose `N_per_kg_milk` goes up by 0.01 kg, which is 10 g. All other columns stay the same. How much does the predicted footprint change?

*Your answer:*

**Quiz scratch space**

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

Click **Runtime > Restart session and run all**. Check that every cell runs. Then click **File > Download > Download .ipynb**. Upload the file to Gradescope. Keep this notebook open for the Canvas quiz.